# 05회차 실습 노트북 — 분류모형(1) — Colab과 전처리

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/05_preprocessing.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/05_preprocessing.html](https://adminmrobo.github.io/fintech_data_analysis/05_preprocessing.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 빈 값 대신 "Unknown"으로 적힌 값

In [2]:
b = load('Revolution/Bank_customers_data.csv')
b['y'] = (b.Attrition_Flag == 'Attrited Customer').astype(int)   # 1 = 이탈 고객
print(b.shape, '이탈 비율:', round(b.y.mean(), 3))
for c in ['Education_Level', 'Income_Category', 'Marital_Status']:
    print(c, (b[c] == 'Unknown').sum())

(10127, 22) 이탈 비율: 0.161
Education_Level 1519
Income_Category 1112
Marital_Status 749


## 2. 이상치: IQR 규칙
제3사분위수(Q3) + 1.5 × IQR 보다 큰 거래 금액을 이상치 후보로 봅니다.

In [3]:
q1, q3 = b.Total_Trans_Amt.quantile([.25, .75]); iqr = q3 - q1
up = q3 + 1.5 * iqr
out = b.Total_Trans_Amt > up
print(f'Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}, 위 경계={up:.0f}, 이상치 후보 {out.sum()}명, 그중 이탈 비율 {b.y[out].mean():.1%}')

Q1=2156, Q3=4741, IQR=2586, 위 경계=8619, 이상치 후보 896명, 그중 이탈 비율 9.8%


이상치 후보의 이탈 비율이 전체(16%)와 크게 다르다면, 지우기 전에 "왜 튀는가"를 먼저 봐야 합니다.

## 3. 대결 재현: Unknown 행 삭제 vs 범주로 유지

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
feat = ['Customer_Age', 'Gender', 'Education_Level', 'Income_Category', 'Marital_Status',
        'Total_Trans_Ct', 'Total_Trans_Amt', 'Total_Relationship_Count', 'Months_Inactive_12_mon']
mask = (b.Education_Level != 'Unknown') & (b.Income_Category != 'Unknown') & (b.Marital_Status != 'Unknown')
tr_i, te_i = train_test_split(np.arange(len(b)), test_size=.25, random_state=RS, stratify=b.y)
tr, te = b.iloc[tr_i], b.iloc[te_i]
def fit(train):
    Xa = pd.get_dummies(pd.concat([train, te])[feat], drop_first=True).astype(float)
    Xtr, Xte = Xa.iloc[:len(train)], Xa.iloc[len(train):]
    s = StandardScaler().fit(Xtr)
    return LogisticRegression(max_iter=3000).fit(s.transform(Xtr), train.y).score(s.transform(Xte), te.y)
print('Unknown 행 삭제 (학습', mask.iloc[tr_i].sum(), '명):', round(fit(tr[mask.iloc[tr_i].values]), 4))
print('Unknown 범주 유지 (학습', len(tr), '명):', round(fit(tr), 4))

Unknown 행 삭제 (학습 5310 명): 0.8606
Unknown 범주 유지 (학습 7595 명): 0.8622


**손으로 한 번** — 아무 모델 없이 "모두 이탈 안 함"이라고 하면 정확도는? 8,500 ÷ 10,127 ≈ 83.9%. 위 두 숫자가 이 기준보다 얼마나 높은지 보세요.